In [2]:
import os
import json
import polars as pl
from pprint import pprint

In [3]:
with open("../../../../../sample_data/response_binance.json", "r") as f:
    response_binance = json.load(f)

with open("../../../../../sample_data/response_bybit.json", "r") as f:
    response_bybit = json.load(f)

In [4]:
class NewsCategory:
    LISTING = "listing"
    DELISTING = "delisting"

In [5]:
def process_binance_announcments(responses):
    result = []
    for response in responses:
        if not response['success']:
            continue

        catalog = response['data']['catalogs'][0]
        type_event = catalog['catalogName']
        if "listing" in type_event: type_event = NewsCategory.LISTING
        elif "delisting" in type_event: type_event = NewsCategory.DELISTING

        for announcement in catalog['articles']:
            announcement['type_event'] = type_event
            result.append(announcement)

    return result
        
    

responses = process_binance_announcments(response_binance)
responses[0]

{'id': 265687,
 'code': 'd2c0b9f4c1e94260b04be48036522cfe',
 'title': 'Notice of Removal of Spot Trading Pairs - 2026-02-13',
 'type': 1,
 'releaseDate': 1770882300000,
 'type_event': 'listing'}

In [6]:
def is_new_listing(result):
    keyword = "New listings"
    fulltext_keyword = keyword.rstrip("s").lower()
    return (
        fulltext_keyword in str(result).lower()
        or keyword in result['topics']
        or keyword in result['category']['key']
        or keyword in result['category']['title']
    )

def is_delisting(result):
    keyword = "Delistings"
    fulltext_keyword = keyword.rstrip("s").lower()
    return (
        fulltext_keyword in str(result).lower()
        or keyword in result['topics']
        or keyword in result['category']['key']
        or keyword in result['category']['title']
    )

In [7]:
def process_bybit_announcments(responses):
    lst = []
    for response in responses:
        if not response['ret_msg']:
            continue
        results = response['result']['hits']
        for result in results:
            if "Derivatives" in result['topics']:
                print("=="*100)
                if is_new_listing(result):
                    type_event = NewsCategory.LISTING
                    pprint(result)

                if is_delisting(result):
                    type_event = NewsCategory.DELISTING

                lst.append(
                    {
                        "type_event": type_event,
                        "releaseDate": result['start_date_timestamp'],
                        "id": result['objectID'],
                        "title": result['title']
                    }
                )    
                

            else:
                continue
    return lst

responses = process_bybit_announcments(response_bybit)

print(responses)

{'_highlightResult': {'title': {'matchLevel': 'none',
                                'matchedWords': [],
                                'value': 'Bybit to convert ESPUSDT Pre-Market '
                                         'Perpetual Contract to standard '
                                         'Perpetual Contract'}},
 'category': {'key': 'new_crypto', 'title': 'New Listings'},
 'date_timestamp': 1770896090,
 'description': '',
 'end_date_timestamp': 1770896090,
 'is_old_url': False,
 'is_top': False,
 'objectID': 'article.bltd3ce068a599f34c8',
 'publish_time': 1770896190,
 'start_date_timestamp': 1770896090,
 'thumbnail_url': 'https://images.contentstack.io/v3/assets/blt8ec5b78e9ea1d11d/bltf13091a0bc954cfa/6927fad42e2f9d0f0dde075d/EN_1600x900_(38).png',
 'title': 'Bybit to convert ESPUSDT Pre-Market Perpetual Contract to standard '
          'Perpetual Contract',
 'topics': ['Derivatives'],
 'url': '/article/bybit-to-convert-espusdt-pre-market-perpetual-contract-to-standard-perp

In [8]:
lst

NameError: name 'lst' is not defined